### data


In [1]:
def get_data():
    return  """ 
| 카테고리명           | 연월     | 판매량(수량) |
|---------------------|---------|-------------|
| Alternative         | 2010-03 | 6           |
| Alternative & Punk  | 2010-01 | 13          |
| Alternative & Punk  | 2010-04 | 3           |
| Alternative & Punk  | 2010-06 | 6           |
| Alternative & Punk  | 2010-08 | 10          |
| Alternative & Punk  | 2010-09 | 6           |
| Alternative & Punk  | 2010-10 | 2           |
| Blues               | 2010-07 | 1           |
| Blues               | 2010-08 | 9           |
| Blues               | 2010-09 | 1           |
| Bossa Nova          | 2010-06 | 2           |
| Classical           | 2010-03 | 4           |
| Classical           | 2010-04 | 10          |
| Comedy              | 2010-02 | 1           |
| Comedy              | 2010-03 | 1           |
| Drama               | 2010-01 | 5           |
| Drama               | 2010-02 | 2           |
| Drama               | 2010-03 | 2           |
| Easy Listening      | 2010-08 | 2           |
| Electronica/Dance   | 2010-03 | 3           |
| Electronica/Dance   | 2010-10 | 1           |
| Heavy Metal         | 2010-09 | 3           |
| Hip Hop/Rap         | 2010-03 | 3           |
| Jazz                | 2010-04 | 3           |
| Jazz                | 2010-06 | 7           |
| Jazz                | 2010-07 | 1           |
| Jazz                | 2010-08 | 1           |
| Jazz                | 2010-09 | 2           |
| Jazz                | 2010-12 | 2           |
| Latin               | 2010-02 | 3           |
| Latin               | 2010-03 | 1           |
| Latin               | 2010-04 | 1           |
| Latin               | 2010-05 | 25          |
| Latin               | 2010-06 | 10          |
| Latin               | 2010-07 | 12          |
| Latin               | 2010-08 | 6           |
| Latin               | 2010-10 | 3           |
| Latin               | 2010-11 | 10          |
| Latin               | 2010-12 | 7           |
| Metal               | 2010-02 | 2           |
| Metal               | 2010-04 | 5           |
| Metal               | 2010-05 | 1           |
| Metal               | 2010-08 | 1           |
| Metal               | 2010-09 | 11          |
| Metal               | 2010-10 | 10          |
| Metal               | 2010-11 | 2           |
| Metal               | 2010-12 | 22          |
| Pop                 | 2010-03 | 7           |
| Pop                 | 2010-05 | 2           |
| R&B/Soul            | 2010-03 | 2           |
| R&B/Soul            | 2010-10 | 5           |
| R&B/Soul            | 2010-11 | 2           |
| Reggae              | 2010-02 | 3           |
| Reggae              | 2010-05 | 4           |
| Rock                | 2010-01 | 10          |
| Rock                | 2010-02 | 21          |
| Rock                | 2010-03 | 5           |
| Rock                | 2010-04 | 14          |
| Rock                | 2010-05 | 4           |
| Rock                | 2010-06 | 13          |
| Rock                | 2010-07 | 24          |
| Rock                | 2010-08 | 7           |
| Rock                | 2010-09 | 14          |
| Rock                | 2010-10 | 16          |
| Rock                | 2010-11 | 24          |
| Rock                | 2010-12 | 5           |
"""


### 코드


In [ ]:
import os, sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("utils"), "..")))
from module.state import *
from module.prompt import *
from module.custom_model import *
from module.base_model import *
from module.tools import *
from module.db_query import *
from langgraph.prebuilt import create_react_agent


In [ ]:
def get_prompt_data_analysis():
    return ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """ 
                You are a helpful assistant.
                You are an intelligent assistant that analyzes the user's request, writes Python code that fulfills the need,and uses the python_repl tool to execute and visualize the code results.
                Instructions:
                - Analyze user's input carefully.
                - Write valid Python code for the task.
                - After writing the code, run it using the python_repl tool.
                - Provide the final output after executing the code.
                """
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{messages}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

In [ ]:
@tool
def generate_python_code(user_input):
    """ 
        You are a tool to write Python code by analyzing your needs
        Write a visualization tool code by analyzing the meaning of the user accurately
        When generating Python code, all text, including title, description, and variables, must be written in English
    """

    llm = get_gemini()
    return llm.invoke(user_input)

In [ ]:
python_repl = PythonAstREPLTool()

# 3. LLM 및 프롬프트 생성
llm = get_gpt()  # 실제 LLM 인스턴스 또는 래퍼 전달
prompt = get_prompt_data_analysis()


# 4. 에이전트 생성
agent_executor = create_react_agent(
    model=llm,
    tools=[generate_python_code, python_repl],
    prompt=prompt
)


config = get_runnable_config(recursion_limit=10, thread_id=get_random_uuid())
messages = get_data()
_messages = "아래 데이터를 하나의 차트에서 선 차트로 비교하게 시각화 해줘" + messages
inputs = {
    "messages": [{"role": "user", "content": _messages}]
}

for a in agent_executor.stream(inputs,config,stream_mode='values'):
    print(a)

